# 技能0 · Day 4 上机：回归分析与概率分布

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **statsmodels** 拟合 OLS 多元线性回归，解读 R²、系数、p值、VIF
2. 用 **statsmodels** 拟合 Logit 逻辑回归，计算倾向性评分（propensity score）
3. 用 **scipy.stats** 拟合正态/二项/泊松分布到真实营销指标，计算概率值
4. 用回归+概率分布计算 LTV（客户终身价值）的点估计和概率区间
5. 用分位数回归分析不同分位上的处理效应（2026前沿）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：statsmodels + scipy.stats + pandas + causaldata。
真实数据：NSW职业培训实验数据（445条，RCT），映射到营销场景。


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 需要 statsmodels, scipy, pandas, causaldata。通常已随 conda/venv 安装。
> causaldata 提供真实RCT数据集，无需手动下载CSV。


In [ ]:
# !pip install statsmodels scipy pandas causaldata -q


## 1. 数据集背景与营销映射

**处理对象**：causaldata NSW职业培训实验数据（445条，RCT，LaLonde 1986）

| NSW字段 | 含义 | 营销映射 | 角色 |
|--------|------|---------|------|
| re78 | 1978年收入($) | 营销后转化金额（Y） | **因变量** |
| re75 | 1975年收入($) | 基线消费（营销前） | 控制变量 |
| age | 年龄 | 用户画像：年龄 | 自变量X |
| educ | 教育年限 | 用户画像：教育水平 | 自变量X |
| treat | 是否参加培训(0/1) | 是否收到营销干预 | 干预变量 |
| black/hisp/marr/nodegree | 人口统计学 | 协变量 | 可选控制 |

**核心分析**：
- OLS回归 re78 ~ age + educ + re75 + treat：量化各因素对转化金额的影响
- Logit回归预测 treat：倾向性评分（连接技能3因果推断）
- 概率分布拟合：正态/二项/泊松分布建模不确定性
- LTV计算：回归+概率分布的综合应用


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 加载真实RCT数据：NSW职业培训实验
from causaldata import nsw_mixtape
df = nsw_mixtape.load_pandas().data

print(f"数据形状: {df.shape}")
print(f"列名: {list(df.columns)}")
print(f"\ntreat分布:\n{df['treat'].value_counts()}")
print(f"\nre78描述统计:\n{df['re78'].describe()}")


## TODO 1：数据加载与探索性分析

**任务**：探索NSW数据的基本结构和分布。

**提示**：
- `df.head()` 查看前5行
- `df.describe()` 查看数值列统计
- `df.groupby('treat')[['re75','re78']].mean()` 对比干预组vs对照组
- `df.isnull().sum()` 检查缺失值

**理论连接**：回归分析前必须做EDA（探索性数据分析），了解数据分布、缺失值、异常值。真实数据的R²往往很低（本Day的OLS R²=0.037），这是真实分析的常态。

**营销映射**：对比treat=1（收到营销干预）vs treat=0（对照组）的re78均值差异，这是A/B测试最基础的分析。


In [ ]:
# 1. 数据加载与探索性分析
head_df = df.head()
desc_stats = df.describe()
group_compare = df.groupby('treat')[['re75', 're78']].mean()
missing_check = df.isnull().sum()

print("=== 前5行 ===")
print(head_df)
print("\n=== 描述统计 ===")
print(desc_stats.round(2))
print("\n=== 干预组 vs 对照组 ===")
print(group_compare)
print("\n=== 缺失值检查 ===")
print(missing_check)


## 2. 回归分析理论回顾

### OLS多元线性回归

`y = b0 + b1*x1 + b2*x2 + ... + bk*xk + e`

关键指标：
- **R²（决定系数）**：模型解释了因变量变异的百分比
- **回归系数p值**：p<0.05表示该变量有统计显著影响
- **VIF（方差膨胀因子）**：VIF>10表示严重多重共线性

### Logit逻辑回归

用于二分类因变量（0/1），输出概率。在本Day中用于计算**倾向性评分**（propensity score）--
即给定协变量，个体接受干预的概率。这是因果推断（技能3）的核心工具。

### 营销映射

| 回归模型 | 营销场景 | statsmodels实现 |
|---------|---------|----------------|
| OLS | 量化各因素对转化金额(re78)的影响 | `sm.OLS(y, sm.add_constant(X)).fit()` |
| Logit | 预测营销干预概率（倾向性评分） | `sm.Logit(y, X).fit()` |


## TODO 2：OLS多元线性回归

**任务**：用 statsmodels 拟合 OLS 回归 re78 ~ age + educ + re75 + treat，解读结果。

**提示**：
- `X = df[['age', 'educ', 're75', 'treat']]` 选取自变量
- `X = sm.add_constant(X)` 添加截距项
- `model = sm.OLS(y, X).fit()` 拟合模型
- `model.summary()` 查看完整结果
- `model.rsquared` / `model.params` / `model.pvalues` 提取关键值
- VIF：`variance_inflation_factor(X.values, i)` 检测共线性

**要求**：
1. 拟合OLS模型，打印summary
2. 提取R²、各系数及其p值
3. 计算VIF检测多重共线性
4. 解读：哪些变量显著？treat（营销干预）的效应是多少？

**营销解读**：treat系数 = 营销干预对转化金额的效应。因为NSW是RCT（随机分配），这个系数可以解释为因果效应。


In [ ]:
# 2. OLS多元线性回归
X_ols = df[['age', 'educ', 're75', 'treat']]
X_ols = sm.add_constant(X_ols)
model_ols = sm.OLS(df['re78'], X_ols).fit()
rsquared = model_ols.rsquared
coefs = model_ols.params
pvalues = model_ols.pvalues

vif_data = pd.DataFrame({
    'variable': X_ols.columns[1:],
    'VIF': [variance_inflation_factor(X_ols.values, i) for i in range(1, X_ols.shape[1])]
})

print("=== OLS回归结果 ===")
print(model_ols.summary())
print(f"\nR² = {rsquared:.6f}")
print(f"调整R² = {model_ols.rsquared_adj:.6f}")
print(f"F p-value = {model_ols.f_pvalue:.2e}")
print("\n=== 系数与p值 ===")
for name in coefs.index:
    print(f"  {name}: coef={coefs[name]:.4f}, p={pvalues[name]:.6f}")
print("\n=== VIF（多重共线性检测，>10为严重）===")
print(vif_data.to_string(index=False))


## TODO 3：Logit逻辑回归与倾向性评分

**任务**：用 statsmodels 拟合 Logit 回归 treat ~ age + educ + re75，计算倾向性评分。

**提示**：
- `X_logit = df[['age', 'educ', 're75']]` 选取自变量（不含treat本身）
- `X_logit = sm.add_constant(X_logit)` 添加截距项
- `y_logit = df['treat']` 因变量是0/1
- `model_logit = sm.Logit(y_logit, X_logit).fit()` 拟合
- `ps = model_logit.predict(X_logit)` 预测倾向性评分

**要求**：
1. 拟合Logit模型，打印summary
2. 计算倾向性评分（propensity score）
3. 分析倾向性评分的分布（mean/std/min/max）
4. 解读：倾向性评分在营销中的用途是什么？

**理论连接**：倾向性评分是因果推断（技能3）的核心工具。在观察数据中（非RCT），干预组和对照组可能存在选择偏差--接受营销干预的用户本身可能更有购买意愿。倾向性评分通过"匹配"相似倾向性的用户来消除这种偏差。


In [ ]:
# 3. Logit逻辑回归与倾向性评分
X_logit = df[['age', 'educ', 're75']]
X_logit = sm.add_constant(X_logit)
y_logit = df['treat']
model_logit = sm.Logit(y_logit, X_logit).fit(disp=0)
propensity_scores = model_logit.predict(X_logit)
ps_mean = propensity_scores.mean()
ps_std = propensity_scores.std()

print("=== Logit回归结果（倾向性评分模型）===")
print(model_logit.summary())
print(f"\n=== 倾向性评分分布 ===")
print(f"  mean = {ps_mean:.6f}")
print(f"  std = {ps_std:.6f}")
print(f"  min = {propensity_scores.min():.6f}")
print(f"  max = {propensity_scores.max():.6f}")
print(f"  median = {propensity_scores.median():.6f}")


## 3. 概率分布理论回顾

### 三种核心概率分布与营销场景

| 分布 | scipy.stats | 营销场景 | 关键参数 |
|------|------------|---------|---------|
| 正态分布 | `norm` | 订单金额（对数后）、客户消费 | mu, sigma |
| 二项分布 | `binom` | 转化/未转化（0/1）、复购 | n, p |
| 泊松分布 | `poisson` | 每日转化次数、客服来电 | mu(lambda) |

### 核心API

| 操作 | 函数 | 用途 |
|------|------|------|
| 拟合 | `norm.fit(data)` | 最大似然估计参数 |
| CDF | `norm.cdf(x, mu, sigma)` | P(X <= x) |
| PPF | `norm.ppf(q, mu, sigma)` | 分位数（逆CDF） |
| PMF | `poisson.pmf(k, mu)` | 离散概率 P(X=k) |
| 区间 | `binom.interval(0.95, n, p)` | 置信区间 |

**营销映射**：概率分布把"转化率是5%"升级为"转化率服从Binomial(n, 0.05)，95%CI为[3.7%, 6.3%]"--不确定性量化是商业决策的基础。


## TODO 4：概率分布拟合与概率计算

**任务**：用 scipy.stats 拟合三种概率分布到NSW真实数据，计算概率值。

**提示**：
- 正态分布：`mu, sigma = stats.norm.fit(df['re78'])` 拟合re78（转化金额）
- 二项分布：`p = df['treat'].mean()` 转化率；`stats.binom.cdf(k, n, p)` 计算概率
- 泊松分布：用 `df.groupby('age')['treat'].sum().mean()` 作为lambda
- 概率计算：`1 - stats.norm.cdf(5000, mu, sigma)` = P(re78 > 5000)
- 分位数：`stats.norm.ppf(0.9, mu, sigma)` = 90th percentile

**要求**：
1. 拟合正态分布到re78，计算P(re78>5000)、P(re78>10000)、90th/95th分位数
2. 用二项分布建模treat：计算p、95% CI、P(10人中>=3人被干预)
3. 用泊松分布建模每年龄组的干预人数：计算lambda、P(X=3)、P(X>=5)
4. （可选）拟合对数正态分布到re78>0（处理长尾）

**营销解读**：P(re78>5000)表示"客户转化金额超过$5000的概率"；二项分布CI告诉你"真实转化率的置信区间"。


In [ ]:
# 4. 概率分布拟合与概率计算
re78 = df['re78']

# (1) 正态分布拟合
norm_mu, norm_sigma = stats.norm.fit(re78)
prob_gt_5000 = 1 - stats.norm.cdf(5000, norm_mu, norm_sigma)
prob_gt_10000 = 1 - stats.norm.cdf(10000, norm_mu, norm_sigma)
pct_90 = stats.norm.ppf(0.9, norm_mu, norm_sigma)
pct_95 = stats.norm.ppf(0.95, norm_mu, norm_sigma)

# (2) 二项分布（treat = 营销干预）
n_total = len(df)
n_treat = int(df['treat'].sum())
p_treat = df['treat'].mean()
binom_ci = stats.binom.interval(0.95, n_total, p_treat)
prob_3_of_10 = 1 - stats.binom.cdf(2, 10, p_treat)

# (3) 泊松分布（每年龄组干预人数）
lambda_pois = df.groupby('age')['treat'].sum().mean()
pois_pmf_3 = stats.poisson.pmf(3, lambda_pois)
pois_gt_5 = 1 - stats.poisson.cdf(4, lambda_pois)

print("=== (1) 正态分布拟合 re78 ===")
print(f"  mu = {norm_mu:.4f}, sigma = {norm_sigma:.4f}")
print(f"  P(re78 > 5000) = {prob_gt_5000:.6f}")
print(f"  P(re78 > 10000) = {prob_gt_10000:.6f}")
print(f"  90th percentile = {pct_90:.4f}")
print(f"  95th percentile = {pct_95:.4f}")

print(f"\n=== (2) 二项分布 treat（营销干预）===")
print(f"  n={n_total}, successes={n_treat}, p={p_treat:.6f}")
print(f"  95% CI for count: [{binom_ci[0]:.0f}, {binom_ci[1]:.0f}]")
print(f"  P(>=3 treated | 10 users, p={p_treat:.4f}) = {prob_3_of_10:.6f}")

print(f"\n=== (3) 泊松分布（每年龄组干预人数）===")
print(f"  lambda = {lambda_pois:.4f}")
print(f"  P(X=3 | lambda={lambda_pois:.2f}) = {pois_pmf_3:.6f}")
print(f"  P(X>=5 | lambda={lambda_pois:.2f}) = {pois_gt_5:.6f}")

# (4) 可选：对数正态分布拟合（处理长尾，过滤零值）
re78_pos = re78[re78 > 0]
ln_shape, ln_loc, ln_scale = stats.lognorm.fit(re78_pos, floc=0)
print(f"\n=== (4) 对数正态分布拟合 re78>0 ({len(re78_pos)} obs) ===")
print(f"  shape(s) = {ln_shape:.6f}, scale = {ln_scale:.4f}")
print(f"  (log-mean = {np.log(ln_scale):.4f}, log-std = {ln_shape:.4f})")


## TODO 5：LTV（客户终身价值）计算

**任务**：用回归+概率分布计算LTV的点估计和概率区间，对比干预组vs对照组。

**LTV公式**：`LTV = 平均客单价(AOV) x 购买频次 x 客户生命周期 x 毛利率`

**NSW数据映射**：
- AOV = re78的均值（人均转化金额）
- 购买频次 = 1次/年（单期观测，简化假设）
- 客户生命周期 = 3年（行业假设）
- 毛利率 = 40%（零售行业典型值）

**提示**：
- `aov = df['re78'].mean()` 人均转化金额
- `ltv_point = aov * frequency * lifetime * gross_margin` 点估计
- `ltv_std = df['re78'].std() * frequency * lifetime * gross_margin` 标准差
- `stats.norm.ppf(0.025, ltv_point, ltv_std)` 95% CI下界
- `stats.norm.ppf(0.975, ltv_point, ltv_std)` 95% CI上界
- 分别计算 treat=1 和 treat=0 的LTV，对比干预效应

**要求**：
1. 计算LTV点估计
2. 计算LTV的95%置信区间和90th分位数
3. 分别计算干预组和对照组的LTV，计算uplift
4. 解读：营销干预的LTV提升是多少？是否值得投入？

**营销解读**：LTV概率区间告诉你"客户终身价值有95%的概率落在[A, B]之间"，比单一数字更有决策价值。


In [ ]:
# 5. LTV（客户终身价值）计算
frequency = 1.0
lifetime = 3.0
gross_margin = 0.40

aov = df['re78'].mean()
ltv_point = aov * frequency * lifetime * gross_margin
ltv_std = df['re78'].std() * frequency * lifetime * gross_margin
ltv_ci_lower = stats.norm.ppf(0.025, ltv_point, ltv_std)
ltv_ci_upper = stats.norm.ppf(0.975, ltv_point, ltv_std)
ltv_p90 = stats.norm.ppf(0.9, ltv_point, ltv_std)

ltv_treated = df[df['treat'] == 1]['re78'].mean() * frequency * lifetime * gross_margin
ltv_control = df[df['treat'] == 0]['re78'].mean() * frequency * lifetime * gross_margin
ltv_uplift = ltv_treated - ltv_control
uplift_pct = (ltv_treated / ltv_control - 1) * 100

print("=== LTV计算（全体客户）===")
print(f"  AOV (mean re78) = ${aov:.2f}")
print(f"  Frequency = {frequency}/year, Lifetime = {lifetime}y, Margin = {gross_margin}")
print(f"  LTV点估计 = ${ltv_point:.2f}")
print(f"  LTV 95% CI = [${ltv_ci_lower:.2f}, ${ltv_ci_upper:.2f}]")
print(f"  LTV 90th percentile = ${ltv_p90:.2f}")

print(f"\n=== 干预组 vs 对照组 ===")
print(f"  LTV (treated) = ${ltv_treated:.2f}")
print(f"  LTV (control) = ${ltv_control:.2f}")
print(f"  LTV uplift = ${ltv_uplift:.2f} ({uplift_pct:.1f}%)")


## TODO 6：分位数回归（2026前沿）

**任务**：用 statsmodels.QuantReg 拟合分位数回归，分析不同分位上的处理效应异质性。

**背景**：OLS回归预测的是条件均值（给定X时Y的期望）。分位数回归预测条件分位数--
如"给定用户画像，转化金额的第75分位数是多少"。对长尾分布（如营销金额）特别有用。

**提示**：
- `model_q = sm.QuantReg(y, X).fit(q=0.5)` 拟合中位数回归
- 分别拟合 q=0.25, 0.5, 0.75
- 对比不同分位上 treat 系数的变化
- `model_q.prsquared` 伪R²

**要求**：
1. 对 q=0.25, 0.5, 0.75 三个分位拟合分位数回归
2. 提取每个分位的 treat 系数和 p值
3. 用OLS模型做预测：对比 treat=1 vs treat=0 的预测re78
4. 解读：treat效应在不同分位上是否不同？这对营销策略有什么启示？

**营销解读**：如果treat在75分位上显著但25分位不显著，说明营销干预对"高价值客户"效果好，对"低价值客户"效果差--这是精准营销的理论基础。


In [ ]:
# 6. 分位数回归（2026前沿）
X_qr = X_ols
y_qr = df['re78']

quantile_results = {}
for q in [0.25, 0.5, 0.75]:
    model_q = sm.QuantReg(y_qr, X_qr).fit(q=q)
    quantile_results[q] = model_q

# OLS预测对比
new_treated = pd.DataFrame({'const': [1], 'age': [25], 'educ': [10], 're75': [1377], 'treat': [1]})
new_control = pd.DataFrame({'const': [1], 'age': [25], 'educ': [10], 're75': [1377], 'treat': [0]})
pred_treated = model_ols.predict(new_treated)
pred_control = model_ols.predict(new_control)
pred_uplift = pred_treated - pred_control

print("=== 分位数回归：treat系数跨分位对比 ===")
for q, model_q in quantile_results.items():
    print(f"  q={q}: treat coef={model_q.params['treat']:.4f} (p={model_q.pvalues['treat']:.6f}), "
          f"educ coef={model_q.params['educ']:.4f} (p={model_q.pvalues['educ']:.6f}), "
          f"Pseudo R²={model_q.prsquared:.6f}")

print(f"\n=== OLS预测对比 ===")
print(f"  典型用户(age=25, educ=10, re75=1377):")
print(f"    treat=1 预测re78 = ${pred_treated.iloc[0]:.2f}")
print(f"    treat=0 预测re78 = ${pred_control.iloc[0]:.2f}")
print(f"    预测uplift = ${pred_uplift.iloc[0]:.2f}")


## 4. 反思与前沿

### 反思问题
1. OLS回归中哪些变量对re78（转化金额）有显著影响？treat（营销干预）的系数是多少？如何解读？
2. R²为什么只有0.037？这说明了什么？真实数据的低R²正常吗？
3. VIF检测发现age和educ的VIF>10，这意味着什么？应该如何处理？
4. 倾向性评分（propensity score）的均值是多少？它在因果推断中有什么用途？
5. 分位数回归中，treat效应在不同分位上如何变化？这对精准营销有什么启示？
6. LTV的95%置信区间为什么这么宽？如何收窄？

### 2026前沿：贝叶斯回归 + 正则化 + 分位数回归

**贝叶斯回归（PyMC/bambi）**：不同于频率派OLS给出系数点估计，贝叶斯回归给出系数的**后验分布**。小样本下通过先验分布提供正则化，更稳健。95%可信区间比频率派置信区间更符合商业决策者的直觉。关键词"贝叶斯"连接2026因果推断前沿。

**Lasso/Ridge正则化（sklearn）**：高维特征下防过拟合，L1做特征选择（系数归零），L2缩小系数。

**分位数回归（QuantReg）**：本Day TODO6已实践。对营销金额的长尾分布特别适用--分别建模普通客户（25分位）和高价值客户（75分位），发现处理效应的异质性。

### 从相关到因果

回归分析揭示的是**相关关系**，不是**因果关系**。NSW是RCT（随机分配），因此treat系数可解释为因果效应。但在观察数据中（非RCT），回归系数只是相关性--从相关到因果需要技能3的工具（do-演算、倾向性评分匹配、工具变量等）。

> "回归是因果推断的基础工具，但回归本身不是因果推断。"
